# 타이타닉으로 배우는 딥러닝 첫걸음
### 모든 인공지능 개발자가 가장 먼저 하는 실습

**Connect AI LAB · 공개판**

---

챗GPT도, 로봇 두뇌도 이 구조 위에 쌓입니다. 두뇌를 하나 만들어 가르치고, 정답을 모르는 문제를 풀게 하는 것.
오늘은 그 첫걸음을 타이타닉 승객 명단으로 합니다.

| | |
|---|---|
| **가르칠 명단** 891명 | 정답(생존)이 있습니다. 이걸로 두뇌를 가르칩니다 |
| **시험 명단** 418명 | 정답이 **없습니다**. 두뇌가 답안지를 씁니다 |
| **답안지.csv** | 순위표에 올리면 점수가 나옵니다. 하루 5번, 전체 30번까지 |

점수는 간단합니다. 418명 중 몇 명을 맞혔는가. 330명이면 79%. **1%는 4명**입니다.
아무것도 안 배우고 전부 「죽는다」로 내면 62%, 이 노트북 그대로 내면 **약 79%** 입니다.

### 오늘 하는 일 여섯 단계

```
 1 · 명단 내려받기          셀 하나로 자동
 2 · 글자를 숫자로          female → 1, 빈칸 → 28
 3 · 어떤 열을 넣을까       표에서 숫자 덩어리 꺼내기        ← 포인트 ①
 4 · 두뇌 만들기            칸 하나짜리 가장 얇은 두뇌       ← 포인트 ②
 5 · 가르치기               891명으로 w 와 b 를 정한다        ← 포인트 ③
 6 · 답안지 만들기 · 제출   418명의 답을 파일로 → 순위표
```

위에서부터 ▶ 를 누르기만 하면 됩니다. 코드를 몰라도 됩니다. 각 셀 위에 **줄마다 무엇을 하는지** 적어 두었습니다.

> 순위표 · https://www.aicitybuilders.com/contest — 회원가입만 하면 누구나 제출할 수 있습니다.

---
# 1 · 명단 내려받기

셀을 실행하면 표 두 개가 인터넷에서 내려옵니다. 파일을 따로 올릴 필요가 없습니다.

| 줄 | 하는 일 |
|---|---|
| `import …` | 쓸 도구를 꺼냅니다. `pd` 는 표를 다루는 도구, `np` 는 숫자 계산, `keras` 는 두뇌를 만드는 도구 |
| `!wget -q -O …` | 인터넷에서 파일을 받아 이름을 붙여 저장합니다. `-q` 는 조용히, `-O` 는 저장할 이름 |
| `pd.read_csv(…)` | csv 파일을 읽어 **표**로 만듭니다. 이제부터 `가르칠명단` 은 891줄짜리 표입니다 |
| `.head()` | 표의 앞 다섯 줄을 보여 줍니다. 잘 읽혔는지 눈으로 확인하는 용도 |

In [ ]:
import numpy as np                          # 숫자 계산
import pandas as pd                         # 표 다루기
import keras                                # 두뇌 만들기

!wget -q -O 가르칠명단.csv "https://drive.google.com/uc?export=download&id=1oDYkq8sd3yD5zBTwGd5V_UcuAvwSn6IN"
!wget -q -O 시험명단.csv "https://drive.google.com/uc?export=download&id=1r4RS_516aMGt5QbyWx5S3qJE9zYEPdi3"

가르칠명단 = pd.read_csv('가르칠명단.csv')   # 891명, 정답(생존) 있음
시험명단   = pd.read_csv('시험명단.csv')     # 418명, 정답 없음

print(f'가르칠 명단 {len(가르칠명단)}명 · 시험 명단 {len(시험명단)}명')
가르칠명단.head()                            # 앞 다섯 줄 보기

### 명단에 있는 열

| 열 | 뜻 | 지금 모양 |
|---|---|---|
| 번호 | 승객 번호. 답안지에 씁니다 | 숫자 |
| 이름 | 승객 이름 | 글자 |
| 성별 | female / male | 글자 |
| 나이 | 나이. 263명은 기록 없음 | 숫자 (일부 NaN) |
| 등급 | 1등석 · 2등석 · 3등석 | 숫자 |
| 요금 | 낸 뱃삯 (파운드) | 숫자 (1명 NaN) |
| 형제배우자 | 같이 탄 형제 · 배우자 수 | 숫자 |
| 부모자녀 | 같이 탄 부모 · 자녀 수 | 숫자 |
| 탑승항 | 탄 항구. S 사우샘프턴 · C 셰르부르 · Q 퀸스타운 | 글자 |
| 생존 | 1 살았다 · 0 죽었다 | **가르칠 명단에만** 있습니다 |

`NaN` 은 "기록이 없다"는 표시입니다. 두뇌는 곱하고 더하는 계산만 하기 때문에 **글자와 빈칸이 있으면 계산 자체가 안 됩니다.** 다음 단계에서 손봅니다.

---
# 2 · 글자를 숫자로

두뇌에는 숫자만 들어갑니다. **두 명단에 똑같이** 손봅니다. 가르칠 때 여성을 1 로 넣었으면 시험 볼 때도 여성은 1 이어야 하니까요.

| 줄 | 하는 일 | 왜 |
|---|---|---|
| `for 명단 in [가르칠명단, 시험명단]:` | 아래 줄들을 두 명단에 차례로 합니다 | 같은 손질을 두 번 적지 않으려고 |
| `.map({'female': 1, 'male': 0})` | 글자를 찾아서 숫자로 바꿉니다 | female 이라는 글자는 곱할 수 없습니다 |
| `.fillna(28)` | 빈칸(NaN)을 28 로 채웁니다 | 28 은 나이가 있는 사람들의 **가운데 나이**. 모르면 보통 사람으로 |
| `.fillna(14)` | 요금 빈칸을 14 로 | 요금의 가운데 값. 빈 사람은 한 명뿐 |
| `.map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)` | 항구를 숫자로, 빈칸은 0(S) | 가장 흔한 항구가 S |

In [ ]:
for 명단 in [가르칠명단, 시험명단]:                                     # 두 명단에 똑같이
    명단['성별']   = 명단['성별'].map({'female': 1, 'male': 0})               # 여성 1, 남성 0
    명단['나이']   = 명단['나이'].fillna(28)                                   # 나이 모르면 28살 (가운데 나이)
    명단['요금']   = 명단['요금'].fillna(14)                                   # 요금 모르면 14 (가운데 요금)
    명단['탑승항'] = 명단['탑승항'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)     # 항구도 숫자로. 모르면 S

print('남은 빈칸 수 — 전부 0 이어야 합니다')
print('가르칠 명단:', int(가르칠명단.isna().sum().sum()), '· 시험 명단:', int(시험명단.isna().sum().sum()))
가르칠명단.head()                                                       # 성별이 0/1 로 바뀌었는지 보세요

---
# 3 · 어떤 열을 두뇌에 넣을까  ← 포인트 ①

처음에는 성별 · 나이 · 등급 세 개만 씁니다. 나중에 열을 **더 넣거나 빼는 것**이 첫 번째 무기입니다.

| 줄 | 하는 일 |
|---|---|
| `쓸열 = [...]` | 두뇌에 넣을 열 이름 목록. 여기만 바꾸면 아래는 자동으로 따라옵니다 |
| `가르칠명단[쓸열].values` | 표에서 그 열들만 뽑아 **숫자 덩어리**로 바꿉니다. 두뇌는 표를 못 읽고 숫자 덩어리만 읽습니다 |
| `입력` | 891명 × 열 개수. 두뇌에 들어가는 쪽 |
| `정답` | 891개의 0 과 1. 두뇌가 맞혀야 하는 쪽 |
| 표준화 | 나이는 0~80, 성별은 0~1 이라 크기가 다릅니다. (값 − 평균) ÷ 퍼짐 으로 비슷한 크기로 맞춥니다. 가르칠 명단 기준으로 계산해 시험 명단에도 **같은 값**을 씁니다 |

`입력.shape` 를 찍으면 `(891, 3)` 처럼 나옵니다. **891명, 한 사람당 숫자 3개**라는 뜻입니다.

In [ ]:
쓸열 = ['성별', '나이', '등급']      # ← 여기에 넣어 보세요: '요금', '형제배우자', '부모자녀', '탑승항'

입력     = 가르칠명단[쓸열].values.astype('float32')   # 표 → 숫자 덩어리 (891명 × 열 개수)
정답     = 가르칠명단['생존'].values.astype('float32')  # 891개의 0 / 1
시험입력 = 시험명단[쓸열].values.astype('float32')     # 시험 명단도 같은 열, 같은 순서

평균, 퍼짐 = 입력.mean(axis=0), 입력.std(axis=0) + 1e-6   # 가르칠 명단 기준
입력     = (입력 - 평균) / 퍼짐                            # 크기 맞추기
시험입력 = (시험입력 - 평균) / 퍼짐                        # 시험 명단에도 같은 값으로

print('두뇌에 넣는 열 :', 쓸열)
print('입력 모양     :', 입력.shape, ' ← (사람 수, 한 사람당 숫자 개수)')

---
# 4 · 두뇌 만들기  ← 포인트 ②

가장 얇은 두뇌입니다. 층을 **하나 더 끼우는 것**이 두 번째 무기입니다.

| 줄 | 하는 일 |
|---|---|
| `keras.Input(shape=(len(쓸열),))` | 들어오는 숫자 개수. 쓸열이 3개면 3, 7개면 7. 3번 셀을 바꾸면 여기가 따라 바뀝니다 |
| `Dense(1, activation='sigmoid')` | 칸 **하나**. 들어온 숫자마다 w 를 곱해 더하고 b 를 더한 뒤, sigmoid 로 0~1 사이 **살 확률**로 만듭니다 |
| `compile(loss=…, optimizer=…)` | 가르치는 방법. `loss` 는 **틀린 정도를 재는 자**, `optimizer` 는 **w 와 b 를 고치는 방법** |
| `Adam(0.003)` | 한 번에 얼마나 고칠지, **보폭**입니다. 크면 빨리 배우지만 흔들리고, 작으면 느립니다 |
| `.summary()` | 두뇌 모양과 정해야 할 숫자 개수(`Param #`)를 보여 줍니다 |

칸 하나가 정해야 할 숫자는 **들어오는 숫자 개수 + 1** 입니다. 열이 3개면 4개.
화살표 자리에 `keras.layers.Dense(8, activation='relu'),` 한 줄을 넣으면 칸 8개짜리 층이 앞에 생깁니다. 더 많이 배울 수 있지만, 더 많이 **외울** 수도 있습니다.

In [ ]:
생존두뇌 = keras.Sequential([
    keras.Input(shape=(len(쓸열),)),                 # 쓸 열 개수만큼 들어옵니다
                                                     # ← 이 자리에  keras.layers.Dense(8, activation='relu'),  를 넣으면 층이 하나 늘어납니다
    keras.layers.Dense(1, activation='sigmoid'),     # 칸 하나. 살 확률을 냅니다
])
생존두뇌.compile(loss='binary_crossentropy', optimizer=keras.optimizers.Adam(0.003), metrics=['accuracy'])
생존두뇌.summary()                                   # Param # = 정해야 할 숫자 개수

---
# 5 · 가르치기  ← 포인트 ③

**몇 번 가르칠지**가 세 번째 무기입니다. 너무 적으면 덜 배우고, 너무 많으면 가르칠 명단만 외웁니다.

| 줄 | 하는 일 |
|---|---|
| `.fit(입력, 정답, epochs=300, validation_split=0.2)` | 891명을 처음부터 끝까지 한 번 보는 게 1 epoch. 300번 되풀이합니다. 20% 는 떼어 두고 **처음 보는 문제**로 씁니다 |
| `verbose=0` | 가르치는 동안 화면에 줄줄이 찍지 않습니다. 보고 싶으면 1 로 |
| `기록.history[...]` | 마지막 epoch 의 연습 점수와 처음 보는 문제 점수 |

**연습 점수**는 두뇌가 이미 본 사람들로 채점한 것이라 높게 나오는 게 보통입니다. **처음 보는 문제 점수**가 진짜에 가깝습니다.
연습은 90% 인데 처음 보는 문제는 75% 라면 891명을 **외운** 겁니다. 시험 문제를 미리 본 학생과 같습니다.

In [ ]:
기록 = 생존두뇌.fit(입력, 정답, epochs=300, batch_size=32, validation_split=0.2, verbose=0)   # ← 300 을 바꿔 보세요

연습점수 = 기록.history['accuracy'][-1] * 100
검증점수 = 기록.history['val_accuracy'][-1] * 100
print(f'연습 점수 {연습점수:.1f}%  ·  처음 보는 문제 점수 {검증점수:.1f}%')
print('진짜 점수는 시험 명단으로 매깁니다. 아래에서 답안지를 만들어 제출하세요.')

처음 보는 문제에서 **78~80%** 가 나오면 정상입니다. 「전부 죽었다」고만 찍어도 62%가 나오니, 두뇌가 뭔가를 배운 겁니다.

---
# 6 · 답안지 만들기 · 제출

시험 명단 418명의 살 확률을 구해서 0.5 를 넘으면 1(산다), 아니면 0(죽는다)으로 적습니다.

| 줄 | 하는 일 |
|---|---|
| `생존두뇌.predict(시험입력)` | 가르친 두뇌에 418명을 넣어 살 확률을 받습니다 |
| `>= 0.5` | 확률이 0.5 이상이면 1, 아니면 0. 이걸 **문턱**이라고 합니다 |
| `pd.DataFrame({'번호': …, '이름': …, '예측': …})` | 답안지 표. **번호**가 있어야 채점기가 누구 답인지 압니다 |
| `.to_csv('답안지.csv', index=False)` | 파일로 저장. 표의 줄 번호는 넣지 않습니다 |
| `files.download(…)` | 내 컴퓨터 다운로드 폴더로 내려받습니다 |

실행하면 `답안지.csv` 가 내려받아집니다. 그 파일을 **손대지 말고 그대로** 순위표에 올리세요.

In [ ]:
예측 = (생존두뇌.predict(시험입력, verbose=0).flatten() >= 0.5).astype(int)   # 살 확률 → 0 / 1

답안지 = pd.DataFrame({'번호': 시험명단['번호'], '이름': 시험명단['이름'], '예측': 예측})
답안지.to_csv('답안지.csv', index=False)
print(f'답안지.csv 저장 — {len(답안지)}명 중 {int(예측.sum())}명을 「산다」로 적었습니다')

from google.colab import files
files.download('답안지.csv')                        # 내 컴퓨터로 내려받습니다
답안지.head()

### 제출 전 확인

채점기가 받아 주는 답안지인지 세 가지를 봅니다. 셋 다 ✓ 가 나와야 합니다.

In [ ]:
확인 = pd.read_csv('답안지.csv')                                   # 저장된 파일을 다시 읽어서 확인합니다
print('✓' if len(확인) == 418 else '✗', f'418명 (지금 {len(확인)}명)')
print('✓' if set(확인['예측'].unique()) <= {0, 1} else '✗', '예측이 0 과 1 뿐')
print('✓' if 확인['번호'].min() == 892 and 확인['번호'].max() == 1309 else '✗', '번호 892 ~ 1309')
print()
print(f'산다 {int(확인["예측"].sum())}명 · 죽는다 {int((확인["예측"] == 0).sum())}명   (참고: 실제 명단은 대략 셋 중 하나가 살았습니다)')

### 순위표에 올리기

1. https://www.aicitybuilders.com/contest — 회원가입하고 로그인합니다 (수강생이 아니어도 됩니다)
2. 방금 내려받은 **답안지.csv** 를 올립니다
3. 바로 채점됩니다. 418명 중 몇 명을 맞혔는지, 몇 등인지, 바로 위 사람까지 몇 %p 인지 나옵니다

🎓 표시는 「나만의 인공지능 · 로컬AI의 정석」 수강생입니다. 같은 표에서 겨룹니다.

---
# 점수 올리는 방법

세 군데를 바꾸고 **3 → 4 → 5 → 6 번 셀을 다시 실행**하면 새 답안지가 나옵니다. 4번 셀을 다시 실행해야 두뇌가 **새로** 만들어집니다.

**레벨 1 · 숫자만 바꾸기**
- 5번 셀 `epochs=300` 을 100, 1000 으로. 적으면 덜 배우고, 많으면 외웁니다. 연습 점수와 처음 보는 문제 점수를 같이 보세요

**레벨 2 · 열 넣기** (보통 여기서 제일 많이 오릅니다)
- 3번 셀 `쓸열` 에 `'요금'` 을 넣어 보세요. 그다음 `'탑승항'`, `'형제배우자'`, `'부모자녀'` 도
- 넣었더니 점수가 떨어지면 다시 빼면 됩니다

**레벨 3 · 층 쌓기**
- 4번 셀의 화살표 자리에 `keras.layers.Dense(8, activation='relu'),` 한 줄
- 8 을 16, 32 로도 바꿔 보고, 같은 줄을 두 번 넣어도 됩니다
- 층을 쌓으면 외우기도 쉬워집니다. 연습 점수만 오르고 처음 보는 문제 점수가 안 오르면 층을 줄이세요

**참고**
- 실행할 때마다 점수가 1~2% 왔다 갔다 하는 건 정상입니다. 시작값이 매번 무작위라서 그렇습니다. 418명에서 1% 는 4명이니, 1등과 3등 차이는 운일 수 있습니다
- 답안지를 여러 번 만들었다면 **연습 점수가 아니라** 제출해서 받은 진짜 점수로 고르세요

## 막히면

| 증상 | 이유와 해결 |
|---|---|
| `NameError` | 위 셀을 건너뛰었습니다. 1번부터 순서대로 |
| `KeyError: '요금'` | 쓸열에 적은 이름이 명단의 열 이름과 다릅니다. 위의 열 표를 보세요 |
| `ValueError: ... shape` | 3번 셀을 바꾼 뒤 4번 셀을 다시 실행하지 않았습니다 |
| 답안지가 안 내려받아짐 | 왼쪽 📁 아이콘 → 답안지.csv 옆 ⋮ → 다운로드 |
| 제출했더니 채점이 안 됨 | 엑셀로 열어 저장하면 번호나 글자가 바뀝니다. 코랩에서 받은 파일을 **그대로** 올리세요 |
| 점수가 62% | 전부 「죽는다」로 낸 겁니다. 2번 셀을 건너뛰었거나, 가르치기 전에 답안지를 만든 경우입니다 |

---
이 실습이 재미있었다면, 여기서 만든 두뇌에 **지식을 연결해 나만의 AI 로 키우는 과정**이 9월에 열려 있습니다.
https://www.aicitybuilders.com/localai

*Connect AI LAB · AI CITY BUILDERS · www.aicitybuilders.com*